# LangChain Pydantic Output Parser Reference

Developer-facing statements defined in `langchain_core.output_parsers.pydantic`.

# `PydanticOutputParser: JsonOutputParser, Generic[TBaseModel]`

Parses JSON model output and validates it against a Pydantic model.

Both Pydantic v2 and Pydantic v1 model classes are supported.

## Fields

```python
pydantic_object: Annotated[type[TBaseModel], SkipValidation()] # Pydantic model used to validate parsed JSON
```

## Constructor

```python
PydanticOutputParser(
    *,
    pydantic_object: Annotated[type[TBaseModel], SkipValidation()], # Pydantic model used to validate parsed JSON
) -> None
```

## Methods

### `parse_result`

Parses the first generation as JSON and validates the resulting object with `pydantic_object`.

```python
@overload
parse_result(
    self,
    result: list[Generation], # Candidate generations for one model input
    *,
    partial: Literal[False] = False, # Require a complete valid result
) -> TBaseModel # Validated Pydantic model instance

@overload
parse_result(
    self,
    result: list[Generation], # Candidate generations for one model input
    *,
    partial: bool = False, # Whether to tolerate incomplete or invalid partial output
) -> TBaseModel | None # Validated model or None for an invalid partial result
```

The method delegates JSON extraction to `JsonOutputParser.parse_result()`.

For a Pydantic v2 model, validation uses `model_validate()`. For a Pydantic v1 model, validation uses `parse_obj()`.

JSON parsing failures and Pydantic validation failures raise `OutputParserException` when `partial=False`.

When Pydantic validation fails, the exception message includes the model name, the parsed JSON object serialized with non-ASCII characters preserved, and the original validation error. The serialized JSON is also stored as `llm_output`.

When `partial=True`, any `OutputParserException` raised during JSON parsing or model validation is suppressed and the method returns `None`. A partial object is returned only after it satisfies the complete Pydantic model schema.

An unsupported Pydantic model version raises `OutputParserException`.

### `parse`

Parses text as JSON and validates it against the configured Pydantic model.

```python
parse(
    self,
    text: str, # Language-model output to parse
) -> TBaseModel # Validated Pydantic model instance
```

The method wraps `text` in a single `Generation` and delegates to `parse_result()`.

### `get_format_instructions`

Returns JSON output instructions containing the configured model's JSON schema.

```python
get_format_instructions(
    self,
) -> str # Pydantic JSON-schema output instructions
```

The model schema is copied before modification. Top-level `"title"` and `"type"` fields are removed when present, and the reduced schema is serialized with `ensure_ascii=False`.

The resulting schema is inserted into an instruction template that explains that the output must be a JSON instance conforming to the supplied schema.

## Properties

### `OutputType`

Returns the configured Pydantic model class.

```python
@property
@override
OutputType(
    self,
) -> type[TBaseModel] # Configured Pydantic model class
```

In [ ]:
from pydantic import BaseModel, Field # Import Pydantic model utilities

from langchain_core.exceptions import OutputParserException # Import the real LangChain parser exception
from langchain_core.output_parsers import PydanticOutputParser # Import the real Pydantic output parser
from langchain_core.outputs import Generation # Import Generation for parse_result


class Product(BaseModel): # Define the expected JSON structure
    name: str # Store the product name
    price: float = Field(gt=0) # Require a positive product price
    in_stock: bool # Store whether the product is available


parser = PydanticOutputParser( # Create the parser
    pydantic_object=Product, # Validate parsed JSON using Product
) # Finish creating the parser

print("Format instructions:") # Display a heading
print(parser.get_format_instructions()) # Display schema-based output instructions

valid_text = '{"name": "Laptop", "price": 59999.0, "in_stock": true}' # Create valid JSON output

product = parser.parse(valid_text) # Parse and validate the JSON text
print("\nParsed product:", product) # Display the validated Product model
print("Product name:", product.name) # Access a validated model field
print("Output type:", parser.OutputType) # Display the configured model class

invoke_result = parser.invoke( # Parse through the runnable interface
    '{"name": "Keyboard", "price": 2499.0, "in_stock": true}', # Provide valid output
    config={"run_name": "parse_product"}, # Name the parser run
) # Finish invoking the parser

print("\nInvoke result:", invoke_result) # Display the runnable result

async_result = await parser.ainvoke( # Parse asynchronously in Jupyter
    '{"name": "Mouse", "price": 999.0, "in_stock": false}', # Provide valid output
    config={"run_name": "parse_product_async"}, # Name the asynchronous parser run
) # Finish asynchronous parsing

print("Async result:", async_result) # Display the asynchronous result

invalid_generation = Generation( # Create invalid model output
    text='{"name": "Monitor", "price": -1000, "in_stock": true}', # Use a negative price
) # Finish creating the generation

partial_result = parser.parse_result( # Parse invalid data in partial mode
    [invalid_generation], # Provide the generation list
    partial=True, # Suppress parser validation errors
) # Finish partial parsing

print("\nPartial invalid result:", partial_result) # Display None

try: # Start complete-result error handling
    parser.parse_result([invalid_generation]) # Parse invalid data as a complete result
except OutputParserException as error: # Catch the LangChain parser exception
    print("Validation error:", error) # Display the validation error
    print("Invalid model output:", error.llm_output) # Display the invalid JSON output